In [ ]:
import pandas as pd
import numpy as np 
from scipy.sparse import hstack
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
import faiss
from sentence_transformers import SentenceTransformer
import ast
import os
import pickle
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 

In [5]:
df = pd.read_csv('../data/final_data/df_web.csv', index_col=0)
df

,title,series,author,rating,description,language,genres,bookFormat,pages,publisher,publishDate,firstPublishDate,awards,coverImg
0,The Hunger Games,The Hunger Games #1,Suzanne Collins,4.33,WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...,en,"['adventure', 'dystopia', 'fantasy']",Hardcover,374,Scholastic Press,2008-09-14 00:00:00,False,True,https://i.gr-assets.com/images/S/compressed.ph...
1,Harry Potter and the Order of the Phoenix,Harry Potter #5,"J.K. Rowling, Mary GrandPré (Illustrator)",4.50,There is a door at the end of a silent corrido...,en,"['adventure', 'childrens', 'classics']",Paperback,870,Scholastic Inc.,2004-09-28 00:00:00,True,True,https://i.gr-assets.com/images/S/compressed.ph...
2,To Kill a Mockingbird,To Kill a Mockingbird,Harper Lee,4.28,The unforgettable novel of a childhood in a sl...,en,"['classics', 'fiction', 'historical']",Paperback,324,Harper Perennial Modern Classics,2006-05-23 00:00:00,True,True,https://i.gr-assets.com/images/S/compressed.ph...
3,Pride and Prejudice,Standalone Novel,"Jane Austen, Anna Quindlen (Introduction)",4.26,Alternate cover edition of ISBN 9780679783268S...,en,"['classics', 'fiction', 'historical']",Paperback,279,Modern Library,2000-10-10 00:00:00,True,False,https://i.gr-assets.com/images/S/compressed.ph...
4,Twilight,The Twilight Saga #1,Stephenie Meyer,3.60,About three things I was absolutely positive.\...,en,"['fantasy', 'fiction', 'paranormal']",Paperback,501,"Little, Brown and Company",2006-09-06 00:00:00,True,True,https://i.gr-assets.com/images/S/compressed.ph...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74779,Beasts & Behemoths (Dungeons & Dragons),Standalone Novel,"Jim Zub, Stacy King, Andrew Wheeler, Official ...",4.04,Study this guide and keep it close at hand--th...,und,['fiction'],Paperback,114,Ten Speed Press,2020-10-20 00:00:00,True,False,http://books.google.com/books/content?id=1toui...
74780,Faculty of Dragon Riders,Standalone Novel,Dmitry Nazarov,4.04,I tamed the Black Dragon!So I thought until I ...,und,['fiction'],Paperback,401,Litres,2022-08-24 00:00:00,True,False,http://books.google.com/books/content?id=QSGFE...
74786,Midnight Delivery Sex,Standalone Novel,Neneko Narazaki,4.00,Mit SNS-Card zum Sammeln in der ersten Auflage...,de,['comics'],Paperback,29,Hayabusa,2021-05-04 00:00:00,True,False,http://books.google.com/books/content?id=s_8_E...
74787,Monster Girl: 2,Standalone Novel,Kazuki Funatsu,4.00,"Dopo il loro incontro, Yatsuki si ritrova ad a...",it,"['comics', 'isekai']",Paperback,216,Edizioni BD,2020-05-01 00:00:00,True,False,http://books.google.com/books/content?id=_yjnE...


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 57305 entries, 0 to 74788
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   title             57305 non-null  object 
 1   series            57305 non-null  object 
 2   author            57305 non-null  object 
 3   rating            57305 non-null  float64
 4   description       57305 non-null  object 
 5   language          57305 non-null  object 
 6   genres            57305 non-null  object 
 7   bookFormat        57305 non-null  object 
 8   pages             57305 non-null  int64  
 9   publisher         57305 non-null  object 
 10  publishDate       57305 non-null  object 
 11  firstPublishDate  57305 non-null  bool   
 12  awards            57305 non-null  bool   
 13  coverImg          57305 non-null  object 
dtypes: bool(2), float64(1), int64(1), object(10)
memory usage: 5.8+ MB


In [7]:
df['genres'].apply(type).value_counts()

genres
<class 'str'>    57305
Name: count, dtype: int64

In [8]:
# Convertir los strings de la columna 'genres' en listas
df['genres'] = df['genres'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else [])

Hacemos Label Encoding de series, language y bookformat.
Los Booleanos firstpublishdate yawards los pasamos a int.
Genres lo pasamos a multilabelbinarizer.
Author hacemos la media de rating.

In [9]:
# Creamos una copia
df_model = df.copy()

# Codificamos variables categóricas con LabelEncoder
cat_cols = ['series', 'language', 'bookFormat']
label_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    df_model[col] = df_model[col].fillna('missing').astype(str) 
    df_model[col] = le.fit_transform(df_model[col])
    label_encoders[col] = le  # Guardamos el encoder

# Convertimos booleanos a enteros
df_model['firstPublishDate'] = df_model['firstPublishDate'].astype(int)
df_model['awards'] = df_model['awards'].astype(int)

# Calculamos el promedio de rating por autor y lo usamos como nueva variable
author_avg = df_model.groupby('author')['rating'].mean()
df_model['author_rating'] = df_model['author'].map(author_avg)
df_model['author_rating'] = df_model['author_rating'].fillna(df_model['rating'].mean())

# Binarizamos géneros (Multi-label one-hot encoding)
mlb = MultiLabelBinarizer()
genres_ohe = mlb.fit_transform(df_model['genres'])
df_genres = pd.DataFrame(genres_ohe, columns=mlb.classes_, index=df_model.index)

# Combinamos con el dataframe principal
df_model = pd.concat([df_model, df_genres], axis=1)

In [10]:
df_model

,title,series,author,rating,description,language,genres,bookFormat,pages,publisher,...,isekai,lgbt,magic,nonfiction,novels,paranormal,romance,suspense,thriller,young adult
0,The Hunger Games,16083,Suzanne Collins,4.33,WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...,14,"[adventure, dystopia, fantasy]",35,374,Scholastic Press,...,0,0,0,0,0,0,0,0,0,0
1,Harry Potter and the Order of the Phoenix,6493,"J.K. Rowling, Mary GrandPré (Illustrator)",4.50,There is a door at the end of a silent corrido...,14,"[adventure, childrens, classics]",62,870,Scholastic Inc.,...,0,0,0,0,0,0,0,0,0,0
2,To Kill a Mockingbird,18386,Harper Lee,4.28,The unforgettable novel of a childhood in a sl...,14,"[classics, fiction, historical]",62,324,Harper Perennial Modern Classics,...,0,0,0,0,0,0,0,0,0,0
3,Pride and Prejudice,13751,"Jane Austen, Anna Quindlen (Introduction)",4.26,Alternate cover edition of ISBN 9780679783268S...,14,"[classics, fiction, historical]",62,279,Modern Library,...,0,0,0,0,0,0,0,0,0,0
4,Twilight,17776,Stephenie Meyer,3.60,About three things I was absolutely positive.\...,14,"[fantasy, fiction, paranormal]",62,501,"Little, Brown and Company",...,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74779,Beasts & Behemoths (Dungeons & Dragons),13751,"Jim Zub, Stacy King, Andrew Wheeler, Official ...",4.04,Study this guide and keep it close at hand--th...,58,[fiction],62,114,Ten Speed Press,...,0,0,0,0,0,0,0,0,0,0
74780,Faculty of Dragon Riders,13751,Dmitry Nazarov,4.04,I tamed the Black Dragon!So I thought until I ...,58,[fiction],62,401,Litres,...,0,0,0,0,0,0,0,0,0,0
74786,Midnight Delivery Sex,13751,Neneko Narazaki,4.00,Mit SNS-Card zum Sammeln in der ersten Auflage...,11,[comics],62,29,Hayabusa,...,0,0,0,0,0,0,0,0,0,0
74787,Monster Girl: 2,13751,Kazuki Funatsu,4.00,"Dopo il loro incontro, Yatsuki si ritrova ad a...",30,"[comics, isekai]",62,216,Edizioni BD,...,1,0,0,0,0,0,0,0,0,0
